# Forecasts

This note book can be run all at once at any point. (Need to add some explanations which explains everyone's models).



## Import and Download Data

In [48]:
# User defined parameters.

# STARTING_DATE = pd.datetime(today)
# FORECAST_HORIZON = 14

In [49]:
# load packages

import requests 
import random
import torch
import numpy as np
import pandas as pd
import warnings
from statsmodels.tools.sm_exceptions import ConvergenceWarning
warnings.simplefilter('ignore', ConvergenceWarning)
from neuralprophet import NeuralProphet
import logging

In [47]:
url = "https://data.cityofnewyork.us/api/views/3q43-55fe/rows.csv?accessType=DOWNLOAD"

response = requests.get(url)
response.raise_for_status()

with open("../scr/data/rat_sightings_data/Updated_Rat_Sightings_NYC.csv", "wb") as f:
    f.write(response.content)

## Citywide

In [5]:
params = dict()
params['apparent_temperature_max'] = 30
params['apparent_temperature_min'] = 5
params['snowfall_sum'] = 1
params['n_lags'] = 16
params['epochs'] = 60 
params['learning_rate'] = 0.2986532324899507
params['batch_size'] = 128 
params['ar_reg'] = 2.132925719127823
params['n_forecasts'] = 14 

In [9]:
rat_sighting = pd.read_csv("../scr/data/rat_sightings_data/Updated_Rat_Sightings_NYC.csv")
rat_sighting.columns = [t.partition('(')[0].strip().lower().replace(' ', '_') for t in rat_sighting.columns] #apply to column headers
rat_sighting['location_type'] = rat_sighting['location_type'].str.strip().str.replace(' ', '_').str.lower()  #apply to location_type column
cols_to_drop = [c for c in rat_sighting.columns if (rat_sighting[c].nunique(dropna=False) == 1)]
rat_sighting = rat_sighting.drop(columns=cols_to_drop)
rat_sighting['created_date'] = pd.to_datetime(rat_sighting['created_date']) 
rat_sighting = rat_sighting.drop(columns='park_borough')
rat_sighting = rat_sighting.drop(columns=['location'])
rs = rat_sighting.copy()
rs['created_date'] = pd.to_datetime(rs['created_date']) 
rs = rs[rs['created_date']>='2020-01-01']
rs = rs.groupby([rs['created_date'].dt.date]).size().reset_index(name='count')
rs.rename(columns={'created_date': 'ds', 'count': 'y'}, inplace=True)

In [11]:
# Weather data
# The forecasted weather values are the naive forecast i.e. we take the last observed day and assume that these are good predictions for the next 14 days.
# Better weather data might help improve the model.

lat, lon = 40.7831, -73.9712
last_date = rs['ds'].max()
start = "2020-01-01"
end   = last_date.strftime("%Y-%m-%d")  # use last date of rs

url = ("https://archive-api.open-meteo.com/v1/archive"
    f"?latitude={lat}&longitude={lon}"
    f"&start_date={start}&end_date={end}"
    "&daily=temperature_2m_max,temperature_2m_min,temperature_2m_mean,"
    "apparent_temperature_max,apparent_temperature_min,apparent_temperature_mean,"
    "precipitation_sum,snowfall_sum"
    "&timezone=America/New_York"
)

response = requests.get(url)
data = response.json()

if 'error' in data:
    nd = pd.read_csv("weatherdata.csv")
    nd['ds'] = pd.to_datetime(nd['date'])
    wd = nd.drop(columns=['date'])
else:
    wd = pd.DataFrame(data["daily"])
    wd["ds"] = pd.to_datetime(wd["time"])
    wd = wd.drop(columns=["time"])

wd = wd.reset_index(drop=True)
future_dates = pd.date_range(start=last_date + pd.Timedelta(days=1),
                             periods=14,
                             freq='D')

last_row = wd.iloc[-1]
wd_14 = pd.DataFrame([last_row.values] * 14, columns=wd.columns)
wd_14['ds'] = future_dates 
wd_14 = wd_14.reset_index(drop=True)

# # weather data in wd_14 should be the same as the last row of wd. 
# # check the displayed dataframe that this is the case

# display(wd.tail(2))
# display(wd_14)


In [12]:
# Suppress cmdstanpy info logs
logging.getLogger("cmdstanpy").setLevel(logging.WARNING)

In [13]:
# set-up regressed features and final set-up for forecasting

regressed_features = ['apparent_temperature_max', 'apparent_temperature_min', 'snowfall_sum']
wd["ds"] = pd.to_datetime(wd["ds"])
wd_14["ds"] = pd.to_datetime(wd_14["ds"])
rs["ds"] = pd.to_datetime(rs["ds"])
rs = rs.merge(wd[['ds'] + regressed_features], on="ds", how="left")

In [14]:
# Fix the seed for the model
# set seeds for reproducibility
seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

# make PyTorch deterministic
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [15]:
forecast_horizon = 14
regressed_features = ['apparent_temperature_max', 'apparent_temperature_min', 'snowfall_sum']

model = NeuralProphet(yearly_seasonality=True, 
                      weekly_seasonality=True, 
                      epochs=params['epochs'], 
                      learning_rate=params['learning_rate'], 
                      batch_size=params['batch_size'], 
                      n_forecasts = params['n_forecasts'], 
                      n_lags = params['n_lags'], 
                      ar_reg = params['ar_reg'], 
                      )
model = model.add_country_holidays(country_name="US")

for col in regressed_features:
    model.add_lagged_regressor(col, n_lags=params[col])

model.fit(rs, freq="D")
future = model.make_future_dataframe(rs, periods=forecast_horizon, n_historic_predictions=28)

for col in regressed_features:
    future[col] = pd.concat([rs[col], wd_14[col]], ignore_index=True)

forecast = model.predict(future)

rs_df = model.get_latest_forecast(forecast, include_previous_forecasts = 14)

WARNING - (NP.forecaster.fit) - When Global modeling with local normalization, metrics are displayed in normalized scale.
INFO - (NP.df_utils._infer_frequency) - Major frequency D corresponds to 99.956% of the data.
INFO - (NP.df_utils._infer_frequency) - Defined frequency is equal to major frequency - D
INFO - (NP.config.init_data_params) - Setting normalization to global as only one dataframe provided for training.
INFO - (NP.utils.set_auto_seasonalities) - Disabling daily seasonality. Run NeuralProphet with daily_seasonality=True to override this.
Missing logger folder: c:\Users\daoke\Documents\GitHub\spring-2026-rat-activity-nyc\notebooks\lightning_logs


Training: 0it [00:00, ?it/s]

INFO - (NP.df_utils._infer_frequency) - Major frequency D corresponds to 99.956% of the data.
INFO - (NP.df_utils._infer_frequency) - Defined frequency is equal to major frequency - D
WARNING - (py.warnings._showwarnmsg) - c:\Users\daoke\anaconda3\Lib\site-packages\neuralprophet\data\split.py:273: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([df, future_df])

INFO - (NP.df_utils.return_df_in_original_format) - Returning df with no ID column
INFO - (NP.df_utils._infer_frequency) - Major frequency D corresponds to 98.611% of the data.
INFO - (NP.df_utils._infer_frequency) - Defined frequency is equal to major frequency - D
INFO - (NP.df_utils._infer_frequency) - Major frequency D corresponds to 98.611% of the data.
INFO - (NP.

Predicting: 18it [00:00, ?it/s]

INFO - (NP.df_utils.return_df_in_original_format) - Returning df with no ID column


In [ ]:
df_wide= rs_df.copy()

# use the last 14 entries of 'origin-0' as the forecast
forecast_horizon = 14
origin_col = 'origin-0' 
last_14_preds = df_wide[origin_col].dropna().tail(forecast_horizon)

# compute corresponding forecast dates
last_14_dates = pd.to_datetime(df_wide['ds'].tail(forecast_horizon))

city_wide_forecast = pd.DataFrame({'ds': last_14_dates, 
                                  'yhat': np.clip(np.round(last_14_preds.astype(float)), 0, None).astype(int)})

# for _, row in forecast_vertical.iterrows():
#     print(f"{row['ds'].date()} -- Prediction: {round(row['yhat'])}.")

In [ ]:
city_wide_forecast

,ds,yhat
14,2026-03-13,29
15,2026-03-14,21
16,2026-03-15,22
17,2026-03-16,47
18,2026-03-17,43
19,2026-03-18,40
20,2026-03-19,42
21,2026-03-20,34
22,2026-03-21,18
23,2026-03-22,23


## Manhattan

In [50]:
params = dict()
params['apparent_temperature_max'] = 54
params['apparent_temperature_min'] = 18
params['snowfall_sum'] = 1
params['n_lags'] = 59 
params['epochs'] = 493 
params['learning_rate'] = 0.003214767890388168 
params['batch_size'] = 220
params['ar_reg'] = 0.5847571241076923
params['n_forecasts'] = 14 

In [51]:
rs = pd.read_csv('../scr/data/rat_sightings_data/Updated_Rat_Sightings_NYC.csv')
rs.columns
rs = rs[['Borough', 'Created Date']]


rs.columns = [t.partition('(')[0].strip().lower().replace(' ', '_') for t in rs.columns] #apply to column headers
rs['created_date'] = pd.to_datetime(rs['created_date']) 
rs = rs[rs['created_date']>='2020-01-01']

rs = rs[rs['borough']=='MANHATTAN']



rs['created_date'] = pd.to_datetime(rs['created_date'])
rs['ds'] = rs['created_date'].dt.date
rs = rs.groupby('ds').size().reset_index(name='y')
rs['ds'] = pd.to_datetime(rs['ds'])

# Date range from 2020-01-01 to the last date in rs
full_range = pd.date_range(start='2020-01-01', end=rs['ds'].max())

# Reindex to include all dates and fill missing y with 0
rs = rs.set_index('ds').reindex(full_range, fill_value=0).rename_axis('ds').reset_index()

In [52]:
# Weather data

lat, lon = 40.7831, -73.9712
last_date = rs['ds'].max()
start = "2020-01-01"
end   = last_date.strftime("%Y-%m-%d")  # use last date of rs

url = (
    "https://archive-api.open-meteo.com/v1/archive"
    f"?latitude={lat}&longitude={lon}"
    f"&start_date={start}&end_date={end}"
    "&daily=temperature_2m_max,temperature_2m_min,temperature_2m_mean,"
    "apparent_temperature_max,apparent_temperature_min,apparent_temperature_mean,"
    "precipitation_sum,snowfall_sum"
    "&timezone=America/New_York"
)

response = requests.get(url)
data = response.json()

if 'error' in data:
    nd = pd.read_csv("weatherdata.csv")
    nd['ds'] = pd.to_datetime(nd['date'])
    wd = nd.drop(columns=['date'])
else:
    wd = pd.DataFrame(data["daily"])
    wd["ds"] = pd.to_datetime(wd["time"])
    wd = wd.drop(columns=["time"])

wd = wd.reset_index(drop=True)
future_dates = pd.date_range(start=last_date + pd.Timedelta(days=1),
                             periods=14,
                             freq='D')

last_row = wd.iloc[-1]
wd_14 = pd.DataFrame([last_row.values] * 14, columns=wd.columns)
wd_14['ds'] = future_dates 
wd_14 = wd_14.reset_index(drop=True)

# Suppress cmdstanpy info logs
logging.getLogger("cmdstanpy").setLevel(logging.WARNING)

# set-up regressed features and final set-up for forecasting

regressed_features = ['apparent_temperature_max', 'apparent_temperature_min', 'snowfall_sum']
wd["ds"] = pd.to_datetime(wd["ds"])
wd_14["ds"] = pd.to_datetime(wd_14["ds"])
rs["ds"] = pd.to_datetime(rs["ds"])
rs = rs.merge(wd[['ds'] + regressed_features], on="ds", how="left")

# Fix the seed for the model
# set seeds for reproducibility
seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

# make PyTorch deterministic
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [53]:
forecast_horizon = 14
regressed_features = ['apparent_temperature_max', 'apparent_temperature_min', 'snowfall_sum']

model = NeuralProphet(yearly_seasonality=True, 
                      weekly_seasonality=True, 
                      epochs=params['epochs'], 
                      learning_rate=params['learning_rate'], 
                      batch_size=params['batch_size'], 
                      n_forecasts = params['n_forecasts'], 
                      n_lags = params['n_lags'], 
                      ar_reg = params['ar_reg'], 
                      )
model = model.add_country_holidays(country_name="US")

for col in regressed_features:
    model.add_lagged_regressor(col, n_lags=params[col])

model.fit(rs, freq="D")
future = model.make_future_dataframe(rs, periods=forecast_horizon, n_historic_predictions=28)

for col in regressed_features:
    future[col] = pd.concat([rs[col], wd_14[col]], ignore_index=True)

forecast = model.predict(future)

rs_df = model.get_latest_forecast(forecast, include_previous_forecasts = 14)

WARNING - (NP.forecaster.fit) - When Global modeling with local normalization, metrics are displayed in normalized scale.
INFO - (NP.df_utils._infer_frequency) - Major frequency D corresponds to 99.956% of the data.
INFO - (NP.df_utils._infer_frequency) - Defined frequency is equal to major frequency - D
INFO - (NP.config.init_data_params) - Setting normalization to global as only one dataframe provided for training.
INFO - (NP.utils.set_auto_seasonalities) - Disabling daily seasonality. Run NeuralProphet with daily_seasonality=True to override this.


Training: 0it [00:00, ?it/s]

INFO - (NP.df_utils._infer_frequency) - Major frequency D corresponds to 99.956% of the data.
INFO - (NP.df_utils._infer_frequency) - Defined frequency is equal to major frequency - D
WARNING - (py.warnings._showwarnmsg) - c:\Users\daoke\anaconda3\Lib\site-packages\neuralprophet\data\split.py:273: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([df, future_df])

INFO - (NP.df_utils.return_df_in_original_format) - Returning df with no ID column
INFO - (NP.df_utils._infer_frequency) - Major frequency D corresponds to 99.01% of the data.
INFO - (NP.df_utils._infer_frequency) - Defined frequency is equal to major frequency - D
INFO - (NP.df_utils._infer_frequency) - Major frequency D corresponds to 99.01% of the data.
INFO - (NP.df

Predicting: 10it [00:00, ?it/s]

INFO - (NP.df_utils.return_df_in_original_format) - Returning df with no ID column


In [54]:
df_wide= rs_df.copy()

# use the last 14 entries of 'origin-0' as the forecast
forecast_horizon = 14
origin_col = 'origin-0' 
last_14_preds = df_wide[origin_col].dropna().tail(forecast_horizon)

# compute corresponding forecast dates
last_14_dates = pd.to_datetime(df_wide['ds'].tail(forecast_horizon))

manhattan_forecast = pd.DataFrame({'ds': last_14_dates, 
                                  'yhat': np.clip(np.round(last_14_preds.astype(float)), 0, None).astype(int)})

In [55]:
manhattan_forecast

,ds,yhat
14,2026-03-13,11
15,2026-03-14,9
16,2026-03-15,12
17,2026-03-16,16
18,2026-03-17,14
19,2026-03-18,17
20,2026-03-19,13
21,2026-03-20,16
22,2026-03-21,11
23,2026-03-22,11


## Brooklyn

Need to say something about why.

In [57]:
params = dict()
params['apparent_temperature_max'] = 54
params['apparent_temperature_min'] = 18
params['snowfall_sum'] = 1
params['n_lags'] = 59 
params['epochs'] = 493 
params['learning_rate'] = 0.003214767890388168 
params['batch_size'] = 220
params['ar_reg'] = 0.5847571241076923
params['n_forecasts'] = 14 

In [56]:
rs = pd.read_csv('../scr/data/rat_sightings_data/Updated_Rat_Sightings_NYC.csv')
rs.columns
rs = rs[['Borough', 'Created Date']]


rs.columns = [t.partition('(')[0].strip().lower().replace(' ', '_') for t in rs.columns] #apply to column headers
rs['created_date'] = pd.to_datetime(rs['created_date']) 
rs = rs[rs['created_date']>='2020-01-01']

rs = rs[rs['borough']=='BROOKLYN']



rs['created_date'] = pd.to_datetime(rs['created_date'])
rs['ds'] = rs['created_date'].dt.date
rs = rs.groupby('ds').size().reset_index(name='y')
rs['ds'] = pd.to_datetime(rs['ds'])

# Date range from 2020-01-01 to the last date in rs
full_range = pd.date_range(start='2020-01-01', end=rs['ds'].max())

# Reindex to include all dates and fill missing y with 0
rs = rs.set_index('ds').reindex(full_range, fill_value=0).rename_axis('ds').reset_index()

In [58]:
# Weather data

lat, lon = 40.7831, -73.9712
last_date = rs['ds'].max()
start = "2020-01-01"
end   = last_date.strftime("%Y-%m-%d")  # use last date of rs

url = (
    "https://archive-api.open-meteo.com/v1/archive"
    f"?latitude={lat}&longitude={lon}"
    f"&start_date={start}&end_date={end}"
    "&daily=temperature_2m_max,temperature_2m_min,temperature_2m_mean,"
    "apparent_temperature_max,apparent_temperature_min,apparent_temperature_mean,"
    "precipitation_sum,snowfall_sum"
    "&timezone=America/New_York"
)

response = requests.get(url)
data = response.json()

if 'error' in data:
    nd = pd.read_csv("weatherdata.csv")
    nd['ds'] = pd.to_datetime(nd['date'])
    wd = nd.drop(columns=['date'])
else:
    wd = pd.DataFrame(data["daily"])
    wd["ds"] = pd.to_datetime(wd["time"])
    wd = wd.drop(columns=["time"])

wd = wd.reset_index(drop=True)
future_dates = pd.date_range(start=last_date + pd.Timedelta(days=1),
                             periods=14,
                             freq='D')

last_row = wd.iloc[-1]
wd_14 = pd.DataFrame([last_row.values] * 14, columns=wd.columns)
wd_14['ds'] = future_dates 
wd_14 = wd_14.reset_index(drop=True)

# Suppress cmdstanpy info logs
logging.getLogger("cmdstanpy").setLevel(logging.WARNING)

# set-up regressed features and final set-up for forecasting

regressed_features = ['apparent_temperature_max', 'apparent_temperature_min', 'snowfall_sum']
wd["ds"] = pd.to_datetime(wd["ds"])
wd_14["ds"] = pd.to_datetime(wd_14["ds"])
rs["ds"] = pd.to_datetime(rs["ds"])
rs = rs.merge(wd[['ds'] + regressed_features], on="ds", how="left")

# Fix the seed for the model
# set seeds for reproducibility
seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

# make PyTorch deterministic
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [59]:
forecast_horizon = 14
regressed_features = ['apparent_temperature_max', 'apparent_temperature_min', 'snowfall_sum']

model = NeuralProphet(yearly_seasonality=True, 
                      weekly_seasonality=True, 
                      epochs=params['epochs'], 
                      learning_rate=params['learning_rate'], 
                      batch_size=params['batch_size'], 
                      n_forecasts = params['n_forecasts'], 
                      n_lags = params['n_lags'], 
                      ar_reg = params['ar_reg'], 
                      )
model = model.add_country_holidays(country_name="US")

for col in regressed_features:
    model.add_lagged_regressor(col, n_lags=params[col])

model.fit(rs, freq="D")
future = model.make_future_dataframe(rs, periods=forecast_horizon, n_historic_predictions=28)

for col in regressed_features:
    future[col] = pd.concat([rs[col], wd_14[col]], ignore_index=True)

forecast = model.predict(future)

rs_df = model.get_latest_forecast(forecast, include_previous_forecasts = 14)

WARNING - (NP.forecaster.fit) - When Global modeling with local normalization, metrics are displayed in normalized scale.
INFO - (NP.df_utils._infer_frequency) - Major frequency D corresponds to 99.956% of the data.
INFO - (NP.df_utils._infer_frequency) - Defined frequency is equal to major frequency - D
INFO - (NP.config.init_data_params) - Setting normalization to global as only one dataframe provided for training.
INFO - (NP.utils.set_auto_seasonalities) - Disabling daily seasonality. Run NeuralProphet with daily_seasonality=True to override this.


Training: 0it [00:00, ?it/s]

INFO - (NP.df_utils._infer_frequency) - Major frequency D corresponds to 99.956% of the data.
INFO - (NP.df_utils._infer_frequency) - Defined frequency is equal to major frequency - D
WARNING - (py.warnings._showwarnmsg) - c:\Users\daoke\anaconda3\Lib\site-packages\neuralprophet\data\split.py:273: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([df, future_df])

INFO - (NP.df_utils.return_df_in_original_format) - Returning df with no ID column
INFO - (NP.df_utils._infer_frequency) - Major frequency D corresponds to 99.01% of the data.
INFO - (NP.df_utils._infer_frequency) - Defined frequency is equal to major frequency - D
INFO - (NP.df_utils._infer_frequency) - Major frequency D corresponds to 99.01% of the data.
INFO - (NP.df

Predicting: 10it [00:00, ?it/s]

INFO - (NP.df_utils.return_df_in_original_format) - Returning df with no ID column


In [60]:
df_wide= rs_df.copy()

# use the last 14 entries of 'origin-0' as the forecast
forecast_horizon = 14
origin_col = 'origin-0' 
last_14_preds = df_wide[origin_col].dropna().tail(forecast_horizon)

# compute corresponding forecast dates
last_14_dates = pd.to_datetime(df_wide['ds'].tail(forecast_horizon))

brooklyn_forecast = pd.DataFrame({'ds': last_14_dates, 
                                  'yhat': np.clip(np.round(last_14_preds.astype(float)), 0, None).astype(int)})

In [61]:
brooklyn_forecast

,ds,yhat
14,2026-03-13,18
15,2026-03-14,10
16,2026-03-15,15
17,2026-03-16,20
18,2026-03-17,19
19,2026-03-18,23
20,2026-03-19,18
21,2026-03-20,18
22,2026-03-21,10
23,2026-03-22,14


## Staten Island

The number of rat sightings in Staten Island on any given day often is 0. Due to the sparsity of the data, we recognize that a daily forecast of two weeks might be unhelpful and so we provide in addition a weekly forecast.

### Daily

In [ ]:
rs = pd.read_csv('../scr/data/rat_sightings_data/Updated_Rat_Sightings_NYC.csv')
rs.columns
rs = rs[['Borough', 'Created Date']]


rs.columns = [t.partition('(')[0].strip().lower().replace(' ', '_') for t in rs.columns] #apply to column headers
rs['created_date'] = pd.to_datetime(rs['created_date']) 
rs = rs[rs['created_date']>='2020-01-01']

rs = rs[rs['borough']=='STATEN ISLAND']

rs['created_date'] = pd.to_datetime(rs['created_date'])
rs['ds'] = rs['created_date'].dt.date
rs = rs.groupby('ds').size().reset_index(name='y')
rs['ds'] = pd.to_datetime(rs['ds'])

# Date range from 2020-01-01 to the last date in rs
full_range = pd.date_range(start='2020-01-01', end=rs['ds'].max())

# Reindex to include all dates and fill missing y with 0
rs = rs.set_index('ds').reindex(full_range, fill_value=0).rename_axis('ds').reset_index()

In [65]:
rs

,ds,y
0,2020-01-01,1
1,2020-01-02,1
2,2020-01-03,1
3,2020-01-04,0
4,2020-01-05,0
...,...,...
2255,2026-03-05,2
2256,2026-03-06,3
2257,2026-03-07,2
2258,2026-03-08,1


### Weekly

## Queens

## Bronx